# Topic 1 — LangChain Fundamentals: Practice Notebook

This notebook is the hands-on companion to `notes/01-langchain-fundamentals.md`.
Read that note first — it explains the *why* and the *math* behind every
section below, with dry runs. This notebook focuses on running the code
yourself.

**How this notebook stays offline and deterministic**: every "model" call in
this notebook uses `GenericFakeChatModel` from `langchain_core`, which returns
pre-scripted responses from a list — no API key, no network call, no Ollama
server required. Section 1 also shows (commented out) how you'd swap in a
real model (`ChatOllama`, `ChatOpenAI`, `ChatAnthropic`) — the rest of every
chain is identical either way, which is the entire point of LangChain's
standard interface.

**Sections**:

1. Models & Messages
2. Prompt Templates
3. Output Parsers
4. LCEL — the `|` operator
5. Memory — **EXERCISE**: `SummarizingChatMessageHistory`
6. Retrievers as Runnables — a tiny keyword-overlap RAG pipeline
7. Tools — **EXERCISE**: define tools + a tool-call dispatch loop

Cells marked **EXERCISE** contain a class/function signature with a docstring
describing exactly what to implement, followed by `# YOUR CODE HERE` and
`pass`. Replace `pass` with your implementation. The `solutions/` copy of this
notebook starts identical to this one — work through it there, and check
`solutions/01-langchain-fundamentals-code-explanation.md` for the canonical
implementation, full walkthroughs, and dry runs once you're done.


In [ ]:
from typing import List

from pydantic import BaseModel, Field

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.tools import tool

print("Imports OK. GenericFakeChatModel stands in for a real chat model below.")


## 1. Models & Messages

A chat model in LangChain is an object whose `.invoke()` method takes a
**list of messages** and returns **one message** back:
`invoke(List[BaseMessage]) -> AIMessage`.

The three message types we'll use:

- `SystemMessage` — instructions about how the model should behave
- `HumanMessage` — something the user said
- `AIMessage` — something the model said

`GenericFakeChatModel` plays back a fixed list of responses, one per call to
`.invoke()`, regardless of the input — perfect for tracing exactly what flows
through a pipeline.


In [ ]:
messages = [
    SystemMessage(content="You are a terse math tutor. Answer with just the number."),
    HumanMessage(content="What is 12 * 4?"),
]
for m in messages:
    print(f"{type(m).__name__:14s} | {m.content}")

# Returns "48." on the first .invoke() call, then "8." on the second call --
# in that fixed order, regardless of what messages it receives.
fake_model = GenericFakeChatModel(messages=iter(["48.", "8."]))

ai_message_1 = fake_model.invoke(messages)
print(ai_message_1)
print("content:", ai_message_1.content)


In [ ]:
# The conversation continues: append the model's reply and a new question
messages.append(ai_message_1)
messages.append(HumanMessage(content="And divided by 6?"))
for m in messages:
    print(f"{type(m).__name__:14s} | {m.content}")

ai_message_2 = fake_model.invoke(messages)
print("content:", ai_message_2.content)


In [ ]:
# The script only had 2 responses -- a third call raises StopIteration
try:
    fake_model.invoke(messages)
except StopIteration:
    print("StopIteration: only 2 responses were scripted.")


### Swapping in a real model

Everything below this point uses `fake_model`. To use a real model instead,
only this one line changes — the rest of every chain stays identical:


In [ ]:
# --- Real model alternatives (commented out -- require API keys / a running server) ---

# from langchain_ollama import ChatOllama
# real_model = ChatOllama(model="llama3.1", temperature=0)

# from langchain_openai import ChatOpenAI
# real_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# from langchain_anthropic import ChatAnthropic
# real_model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)


## 2. Prompt Templates

`PromptTemplate` fills `{placeholders}` into a plain string.
`ChatPromptTemplate` fills placeholders into a *list of messages* — the shape
a chat model expects.


In [ ]:
translate_template = PromptTemplate.from_template(
    "Translate the following English text to {language}:\n\n{text}"
)
print("input_variables:", translate_template.input_variables)

formatted = translate_template.format(language="French", text="Good morning")
print(formatted)


### `ChatPromptTemplate` — placeholders inside a list of messages


In [ ]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}."),
    ("human", "{question}"),
])
print("input_variables:", chat_template.input_variables)

prompt_value = chat_template.invoke({"persona": "terse math tutor", "question": "What is 9*9?"})
print(type(prompt_value))
for m in prompt_value.to_messages():
    print(f"{type(m).__name__:14s} | {m.content}")


### Partial variables — bake in a value now, fill the rest later


In [ ]:
tutor_template = chat_template.partial(persona="terse math tutor")
print("input_variables:", tutor_template.input_variables)

prompt_value_2 = tutor_template.invoke({"question": "What is 7*8?"})
for m in prompt_value_2.to_messages():
    print(f"{type(m).__name__:14s} | {m.content}")


### Few-shot prompt templates — show examples instead of just describing the task


In [ ]:
example_prompt = PromptTemplate.from_template("Input: {input}\nOutput: {output}")

examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
]

few_shot_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Input: {adjective}\nOutput:",
    input_variables=["adjective"],
)

print(few_shot_template.format(adjective="fast"))


## 3. Output Parsers

Output parsers convert the `AIMessage` a model returns into the type your
application wants: a plain string, a dict, or a validated object.


In [ ]:
str_parser = StrOutputParser()
ai_msg = AIMessage(content="The capital of France is Paris.")

parsed = str_parser.invoke(ai_msg)
print(repr(parsed))
print("isinstance(parsed, str):", isinstance(parsed, str))


In [ ]:
json_parser = JsonOutputParser()
ai_msg_json = AIMessage(content='{"title": "Inception", "rating": 9}')

parsed_json = json_parser.invoke(ai_msg_json)
print(parsed_json)
print(type(parsed_json))


### `PydanticOutputParser` — validated objects + format instructions


In [ ]:
class MovieReview(BaseModel):
    title: str = Field(description="The movie's title")
    rating: int = Field(description="Rating out of 10")
    summary: str = Field(description="One-sentence summary")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

# Format instructions sent to the model (truncated)
print(pydantic_parser.get_format_instructions()[:300], "...")

valid_json = '{"title": "Inception", "rating": 9, "summary": "A heist within dreams."}'
review = pydantic_parser.invoke(AIMessage(content=valid_json))
print(review)
print("review.title :", review.title)
print("review.rating:", review.rating)


In [ ]:
# Valid JSON, but "rating" is a string instead of an int -- Pydantic rejects it
bad_json = '{"title": "Inception", "rating": "nine", "summary": "A heist within dreams."}'
try:
    pydantic_parser.invoke(AIMessage(content=bad_json))
except Exception as e:
    print(type(e).__name__, "raised:")
    print(e)


## 4. LCEL — the `|` operator

`chain = a | b | c` builds a `Runnable` whose `.invoke(x)` is
`c.invoke(b.invoke(a.invoke(x)))`. Below we trace each stage by hand first,
then build the same pipeline with `|`.


In [ ]:
inputs = {"persona": "terse math tutor", "question": "What is 9*9?"}

stage1 = chat_template.invoke(inputs)
print("stage1:", stage1.to_messages())

# A fresh fake model with one scripted reply for this trace
fake_model_lcel = GenericFakeChatModel(messages=iter(["81."]))
stage2 = fake_model_lcel.invoke(stage1)
print("stage2:", stage2)

stage3 = str_parser.invoke(stage2)
print("stage3:", repr(stage3))


In [ ]:
# The same pipeline written with LCEL's | operator
fake_model_lcel_2 = GenericFakeChatModel(messages=iter(["81."]))
chain = chat_template | fake_model_lcel_2 | str_parser

result = chain.invoke({"persona": "terse math tutor", "question": "What is 9*9?"})
print(repr(result))
print("matches manual trace:", result == stage3)


### `RunnableLambda` — wrapping a plain function


In [ ]:
shout = RunnableLambda(lambda text: text.upper() + "!")

fake_model_lcel_3 = GenericFakeChatModel(messages=iter(["81."]))
chain_with_shout = chat_template | fake_model_lcel_3 | str_parser | shout

result_shout = chain_with_shout.invoke({"persona": "terse math tutor", "question": "What is 9*9?"})
print(repr(result_shout))


### `RunnableParallel` and `RunnablePassthrough`


In [ ]:
branch = RunnableParallel({
    "original": RunnablePassthrough(),
    "upper": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(lambda x: len(x)),
})

print(branch.invoke("hello"))


## 5. Memory

A chat model is stateless — "memory" means *your application* manages the
growing list of messages. `InMemoryChatMessageHistory` is the simplest
container: a list with `.add_message()` and `.messages`.


In [ ]:
history = InMemoryChatMessageHistory()
history.add_message(HumanMessage(content="What is self-attention?"))
history.add_message(AIMessage(content="It lets each token weigh every other token."))
history.add_message(HumanMessage(content="And LoRA?"))
history.add_message(AIMessage(content="It fine-tunes via small low-rank matrices."))

print("len(history.messages) =", len(history.messages))
for m in history.messages:
    print(f"{type(m).__name__:14s} | {m.content}")


### EXERCISE — `SummarizingChatMessageHistory`

Implement a chat message history that automatically summarizes old messages
once the history grows beyond a limit. Read the docstring carefully — it
specifies the exact splitting and replacement behavior.


In [ ]:
class SummarizingChatMessageHistory(InMemoryChatMessageHistory):
    """
    A chat message history that automatically summarizes old messages once
    the history grows beyond `max_messages`.

    This class extends `InMemoryChatMessageHistory`, which already provides
    `self.messages: List[BaseMessage]` and a working `add_message` method
    that simply appends to that list.

    `InMemoryChatMessageHistory` (like everything in langchain-core) is a
    Pydantic model, so its constructor is generated automatically from
    class-level type-annotated fields -- you do NOT write an `__init__`.
    Declare three fields:

        model: BaseChatModel -- used to generate the summary
        max_messages: int -- summarization triggers once
            len(self.messages) exceeds this
        keep_last: int = 2 -- how many of the most recent messages to leave
            untouched when summarizing

    With these declared, `SummarizingChatMessageHistory(model=..., max_messages=4)`
    works automatically (Pydantic builds the constructor for you).

    Override `add_message(message)` with this behavior:

        1. Call the parent class's `add_message` to append `message` to
           `self.messages`, exactly as it would normally.

        2. If `len(self.messages) <= self.max_messages`, do nothing else --
           return.

        3. Otherwise (the history is now too long), summarize:

           a. Split `self.messages` into:
                - `old_messages`    = self.messages[:-self.keep_last]
                - `recent_messages` = self.messages[-self.keep_last:]

           b. Build a single summarization request: a list containing one
              HumanMessage whose content asks the model to summarize the
              conversation so far in 1-2 sentences, followed by the text of
              `old_messages` -- one line per message, formatted as
              "<ClassName>: <content>", e.g.:
                  "Summarize the following conversation in 1-2 sentences:\n"
                  "HumanMessage: What is self-attention?\n"
                  "AIMessage: It lets each token weigh every other token."

           c. Call `self.model.invoke(...)` with that single-message list.
              This returns an AIMessage; its `.content` is the summary text.

           d. Replace `self.messages` (in place, e.g. via `self.messages =`)
              with:
                  [SystemMessage(content=f"Summary of earlier conversation: {summary_text}")]
                  + recent_messages

    Returns:
        None. Like the parent class's add_message, this mutates
        self.messages in place.

    Math note: this keeps the message list bounded by roughly
    `max_messages` -- every time it would exceed that bound, it collapses
    `len(self.messages) - keep_last` messages down to exactly 1
    SystemMessage, so the list shrinks back to `keep_last + 1` messages.
    """

    # YOUR CODE HERE -- declare the `model`, `max_messages`, `keep_last` fields

    def add_message(self, message):
        # YOUR CODE HERE
        pass


In [ ]:
# The fake model will be asked to summarize exactly once (when the 5th
# message pushes the history from 4 -> 5, exceeding max_messages=4).
summarizing_model = GenericFakeChatModel(
    messages=iter(["User asked about self-attention and LoRA; assistant explained both."])
)

summarizing_history = SummarizingChatMessageHistory(
    model=summarizing_model, max_messages=4, keep_last=2
)

turns = [
    HumanMessage(content="What is self-attention?"),
    AIMessage(content="It lets each token weigh every other token."),
    HumanMessage(content="And LoRA?"),
    AIMessage(content="It fine-tunes via small low-rank matrices."),
    HumanMessage(content="What about BPE?"),
]

for i, turn in enumerate(turns):
    summarizing_history.add_message(turn)
    print(f"after message {i + 1} ({type(turn).__name__}: {turn.content!r}):")
    print("  len(messages) =", len(summarizing_history.messages))
    for m in summarizing_history.messages:
        print(f"    {type(m).__name__:14s} | {m.content}")

print()
print("expected after message 5: 3 messages --")
print("  [SystemMessage(summary), AIMessage('...LoRA...'), HumanMessage('What about BPE?')]")


## 6. Retrievers as Runnables

A retriever is a `Runnable` with `.invoke(query: str) -> List[Document]`.
Below we build a tiny corpus and a hand-written `SimpleKeywordRetriever` that
scores documents by word overlap with the query — fully deterministic, no
embeddings needed.


In [ ]:
corpus = [
    Document(page_content="Self-attention computes a weighted average of value vectors", metadata={"id": "A"}),
    Document(page_content="LoRA fine-tunes a model using small low-rank matrices", metadata={"id": "B"}),
    Document(page_content="Byte pair encoding merges frequent character pairs into subwords", metadata={"id": "C"}),
    Document(page_content="Retrieval augmented generation retrieves documents before generating an answer", metadata={"id": "D"}),
    Document(page_content="Direct preference optimization trains a model directly on preference pairs", metadata={"id": "E"}),
]

for doc in corpus:
    print(f"[{doc.metadata['id']}] {doc.page_content}")


In [ ]:
class SimpleKeywordRetriever(BaseRetriever):
    """
    A retriever that scores documents by counting how many words they share
    with the query (case-insensitive, exact-match only).

    Attributes:
        documents: List[Document] -- the corpus to search over
        k: int -- how many top-scoring documents to return (default 1)

    Math note: relevance(query, doc) = |words(query) intersect words(doc)|,
    where words(s) = set(s.lower().split()). Documents are sorted by this
    score, descending, and the top k are returned.
    """
    documents: List[Document]
    k: int = 1

    def _get_relevant_documents(self, query, *, run_manager=None):
        query_words = set(query.lower().split())
        scored = []
        for doc in self.documents:
            doc_words = set(doc.page_content.lower().split())
            overlap = len(query_words & doc_words)
            scored.append((overlap, doc))
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return [doc for score, doc in scored[: self.k]]


retriever = SimpleKeywordRetriever(documents=corpus, k=1)

query = "How does attention compute weighted values?"
print("query words:", set(query.lower().split()))

for doc in retriever.invoke(query):
    print(f"retrieved [{doc.metadata['id']}]: {doc.page_content}")


### Composing a tiny RAG chain with `RunnableParallel`


In [ ]:
def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

format_docs_runnable = RunnableLambda(format_docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY this context:\n{context}"),
    ("human", "{question}"),
])

parallel_stage = RunnableParallel({
    "context": retriever | format_docs_runnable,
    "question": RunnablePassthrough(),
})

intermediate = parallel_stage.invoke(query)
print("context :", intermediate["context"])
print("question:", intermediate["question"])


In [ ]:
rag_model = GenericFakeChatModel(messages=iter([
    "Attention computes a weighted average of value vectors, using weights "
    "derived from query-key dot products."
]))

rag_chain = parallel_stage | rag_prompt | rag_model | StrOutputParser()

answer = rag_chain.invoke(query)
print(answer)


## 7. Tools

`@tool` turns a Python function into a `BaseTool` carrying a name,
description, and argument schema — metadata that gets shown to a model so it
can *request* that tool be called via `AIMessage.tool_calls`. Your code is
responsible for actually calling the function — the model never executes
anything itself.


In [ ]:
@tool
def word_count(text: str) -> int:
    """Count the number of whitespace-separated words in text."""
    return len(text.split())

print("name       :", word_count.name)
print("description:", word_count.description)
print("args schema:", word_count.args)
print("invoke:", word_count.invoke({"text": "hello world"}))


### EXERCISE — define two tools and a tool-call dispatch loop

Implement `add_numbers`, `reverse_text`, and `execute_tool_calls` below. The
test cell after them constructs a hand-made `AIMessage` with `.tool_calls`
(simulating what a real model would return) and runs your dispatcher against
it.


In [ ]:
@tool
def add_numbers(a: int, b: int) -> int:
    """
    EXERCISE: Add two integers together and return the sum.

    Args:
        a: the first integer
        b: the second integer

    Returns:
        The sum a + b, as an int.
    """
    # YOUR CODE HERE
    pass


@tool
def reverse_text(text: str) -> str:
    """
    EXERCISE: Reverse a string.

    Args:
        text: the string to reverse

    Returns:
        `text` with its characters in reverse order, e.g. "hello" -> "olleh".
    """
    # YOUR CODE HERE
    pass


In [ ]:
def execute_tool_calls(ai_message, tools_by_name):
    """
    EXERCISE: Execute every tool call requested in `ai_message` and return
    the results as a list of ToolMessage objects.

    Args:
        ai_message: an AIMessage whose `.tool_calls` attribute is a list of
            dicts, each shaped like:
                {"name": <tool name str>, "args": <dict of kwargs>, "id": <call id str>}
        tools_by_name: dict mapping tool name (str) -> the @tool-decorated
            function (a BaseTool), e.g.
                {"add_numbers": add_numbers, "reverse_text": reverse_text}

    Behavior:
        For each tool_call dict in ai_message.tool_calls, in order:
          1. Look up the tool function: tool_fn = tools_by_name[tool_call["name"]]
          2. Run it: result = tool_fn.invoke(tool_call["args"])
          3. Wrap the result: ToolMessage(content=str(result), tool_call_id=tool_call["id"])

    Returns:
        List[ToolMessage], one per entry in ai_message.tool_calls, in the
        same order.
    """
    # YOUR CODE HERE
    pass


In [ ]:
tools_by_name = {"add_numbers": add_numbers, "reverse_text": reverse_text}

# A hand-constructed AIMessage, simulating what a real model would return
# after deciding to call both tools in one turn.
fake_tool_call_message = AIMessage(
    content="",
    tool_calls=[
        {"name": "add_numbers", "args": {"a": 12, "b": 30}, "id": "call_001"},
        {"name": "reverse_text", "args": {"text": "hello"}, "id": "call_002"},
    ],
)

print("tool_calls:", fake_tool_call_message.tool_calls)

tool_messages = execute_tool_calls(fake_tool_call_message, tools_by_name)
print("results:", tool_messages)

print()
print("expected:")
print("  ToolMessage(content='42', tool_call_id='call_001')")
print("  ToolMessage(content='olleh', tool_call_id='call_002')")


## Summary

You've now used every piece from `notes/01-langchain-fundamentals.md`:
messages and a fake chat model, prompt templates (plain, chat, partial,
few-shot), output parsers (str / json / pydantic), LCEL composition with `|`,
`RunnableLambda` / `RunnableParallel` / `RunnablePassthrough`, a summarizing
chat memory (exercise), a keyword-overlap retriever wired into a RAG chain,
and a tool-call dispatch loop (exercise).

Once you've completed the two exercises (Section 5 and Section 7), check
`solutions/01-langchain-fundamentals-code-explanation.md` for the canonical
implementations, full walkthroughs, and dry runs against the exact numbers
used in this notebook.

**Next**: Topic 2 — LangGraph, where the flat `|` chains here become a
stateful graph that can loop (e.g. repeat the tool-call round trip from
Section 7 until the model is satisfied).
